%% [markdown]
# GPN-Star inference parity — CPU / CUDA / Trainium  (DEV mode)
Runs the vendored **full model** (`GPNStarForMaskedLM`, MSA encoder + MLM head) on every available
device and checks the output logits match. CPU is the reference; `cuda` auto-skips when absent; `neuron`
runs on the Trainium core. Pin a free core:
`NEURON_RT_VISIBLE_CORES=0 jupyter nbconvert --to notebook --execute 01_inference_parity.ipynb`.

In [1]:
# %%
import os
os.environ.setdefault("NEURON_RT_VISIBLE_CORES", "0")
import sys
sys.path.insert(0, os.path.abspath("../src"))
import torch
import torch.nn.functional as F
import gpn_star_reference as R

[W709 03:29:34.049123109 OperatorEntry.cpp:208] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::gather.out(Tensor self, int dim, Tensor index, *, bool sparse_grad=False, Tensor(a!) out) -> Tensor(a!)
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: PrivateUse1
  previous kernel: registered at /pytorch/build/aten/src/ATen/RegisterCPU_3.cpp:7637
       new kernel: registered at NeuronDispatcher.cpp:0 (function operator())
/home/user/miniconda3/envs/gpn-vep/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def devices():
    devs = ["cpu"]
    if torch.cuda.is_available():
        devs.append("cuda")
    try:
        import torch_neuronx  # noqa: F401
        devs.append("neuron")
    except Exception as e:
        print("neuron unavailable:", e)
    return devs

In [3]:
DEVICES = devices()
print("torch", torch.__version__, "| devices:", DEVICES, "| model:", R.MODEL_NAME)

torch 2.11.0+cpu | devices: ['cpu', 'neuron'] | model: songlab/gpn-star-tair10-b18-25m


In [4]:
# %% [markdown]
# ## Run the full model on each device
# %%
def run(device):
    model = R.load(device)
    inputs = R.build_inputs()
    with torch.no_grad():
        out = model(*[t.to(device) for t in inputs])
    if device == "neuron":
        import torch_neuronx; torch_neuronx.synchronize()
    return tuple(t.detach().float().cpu() for t in out)

In [5]:
results = {d: run(d) for d in DEVICES}
for name, t in zip(R.OUTPUT_ORDER, results["cpu"]):
    print(f"cpu {name:8s} shape={tuple(t.shape)}")

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 110700.11it/s]

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 129720.74it/s]


/home/user/miniconda3/envs/gpn-vep/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 78479.70it/s]

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 112347.43it/s]

cpu logits   shape=(1, 64, 1, 6)


In [6]:
# %% [markdown]
# ## Check every device matches CPU (per-output cosine + max-abs)
# %%
def cos(a, b): return F.cosine_similarity(a.flatten(), b.flatten(), dim=0).item()

In [7]:
ref, all_ok = results["cpu"], True
for d in DEVICES:
    if d == "cpu":
        continue
    print(f"\n{d} vs cpu:")
    for name, a, b in zip(R.OUTPUT_ORDER, ref, results[d]):
        c = cos(a, b); mab = (a - b).abs().max().item(); ok = c >= 0.99
        all_ok = all_ok and ok
        print(f"  {name:8s} cosine={c:.6f}  max-abs={mab:.3e}  {'OK' if ok else 'FAIL'}")


neuron vs cpu:
  logits   cosine=1.000000  max-abs=1.144e-05  OK


In [8]:
print("\nINFERENCE PARITY:", "PASS" if all_ok else "FAIL")
assert all_ok, "full-model outputs diverged across devices"


INFERENCE PARITY: PASS
